# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q('''
SELECT
  t.title,
  a.name AS artist_name,
  a.country
FROM
  tracks AS t
JOIN
  artists AS a
ON
  t.artist_id = a.artist_id;
''')

,title,artist_name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q('''
SELECT
  genre,
  AVG(seconds) AS average_track_length
FROM
  tracks
GROUP BY
  genre
ORDER BY
  average_track_length DESC
LIMIT 1;
''')

,genre,average_track_length
0,Electronic,287.5


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q('''
SELECT
  user,
  COUNT(play_id) AS total_plays,
  COUNT(DISTINCT track_id) AS distinct_tracks_played
FROM
  plays
GROUP BY
  user;
''')

,user,total_plays,distinct_tracks_played
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q('''
SELECT
  t.track_id,
  t.title
FROM
  tracks AS t
LEFT JOIN
  plays AS p
ON
  t.track_id = p.track_id
WHERE
  p.play_id IS NULL;
''')

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q('''
SELECT
  a.name AS artist_name,
  SUM(t.seconds) AS total_listening_seconds,
  ROUND(SUM(t.seconds) * 1.0 / 60, 1) AS total_listening_minutes
FROM
  artists AS a
JOIN
  tracks AS t
ON
  a.artist_id = t.artist_id
JOIN
  plays AS p
ON
  t.track_id = p.track_id
GROUP BY
  a.name
ORDER BY
  total_listening_seconds DESC;
''')

,artist_name,total_listening_seconds,total_listening_minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [9]:
q('''
SELECT
  track_id,
  title
FROM
  tracks
WHERE
  genre IS NULL;
''')

,track_id,title
0,18,Untitled Demo


### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [10]:
q('''
SELECT
  played_on AS play_date,
  COUNT(play_id) AS total_plays,
  COUNT(DISTINCT user) AS distinct_users_active
FROM
  plays
GROUP BY
  played_on
ORDER BY
  played_on ASC;
''')

,play_date,total_plays,distinct_users_active
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [11]:
q1 = q('''
SELECT
  t.title,
  a.name AS artist_name,
  a.country
FROM
  tracks AS t
JOIN
  artists AS a
ON
  t.artist_id = a.artist_id;
''')

q4 = q('''
SELECT
  t.track_id,
  t.title
FROM
  tracks AS t
LEFT JOIN
  plays AS p
ON
  t.track_id = p.track_id
WHERE
  p.play_id IS NULL;
''')

q3 = q('''
SELECT
  user,
  COUNT(play_id) AS total_plays,
  COUNT(DISTINCT track_id) AS distinct_tracks_played
FROM
  plays
GROUP BY
  user;
''')

assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['total_plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

I think Question 4 was a bit challenging initially because I couldn't figure out exactly how the different JOIN types work when it came to finding missing data. I initially tried to JOIN track and plays, then filter for a NULL play ID. Then I realized this wouldn't return anything because it only returns rows when there is a match in the first place in both tables. The NULLs would thus be excluded from the outset. I then read the hint suggesting I use LEFT JOIN, which gave the solution I needed!